# Stage 1: the AQx-2 pair structures, from the crystal

Extracts the two stacked dimer motifs of AQx-2 from the experimental
single-crystal structure, CCDC 2161927 (*Nat. Commun.* **14**, 5079 (2023)).

**Why the crystal rather than a built dimer.** AQx-2 is banana-shaped, so it
does not stack by translating a copy along a plane normal - it stacks by
nesting, offset and matched to its own curvature. Constructing a pair in the
gas phase produces clashes or edge-only overlap for exactly that reason. The
crystal already contains the answer, and the paper reports four overlaps per
molecule: J-aggregation between end groups at 3.341 A, and H-aggregation
between backbones at 3.424 A. This notebook reproduces both.

**The alkyl chains are dropped.** They are severely disordered in this
structure (U up to 0.657, two-component disorder held by DFIX restraints), so
their coordinates are unreliable and clustering on them fuses neighbouring
molecules into one blob. They are also what you would truncate for a coupling
calculation anyway - the frontier orbitals live on the conjugated core - so the
core is kept and its two attachment points are capped with hydrogen.

## Input

`2161927.cif` in this directory. Download it from the CCDC entry if it is not
already here.

## Output

`crystal_dimers/H_backbone_dimer.xyz` and `crystal_dimers/J_end_group_dimer.xyz`
- both molecules of each pair, monomer A first then monomer B. That ordering is
what stage 2 relies on, and the last cell checks it.

Needs `numpy` and `scipy`.

In [ ]:
# ---- Settings -----------------------------------------------------------
import os
import re
from collections import Counter

import numpy as np
from scipy.spatial import cKDTree

CIF = "2161927.cif"
OUTPUT_DIR = "crystal_dimers"

RCOV = {"H": 0.31, "C": 0.76, "N": 0.71, "O": 0.66, "F": 0.57, "S": 1.05}
SOLVENT = {"Cl1", "Cl2", "Cl3", "C51", "H51"}   # disordered chloroform
MINOR = {"2", "-2"}                             # minor disorder components
CAP = {"N": 1.01, "C": 1.09}                    # X-H capping bond lengths
CONTACT = 4.5                                   # A, defines a neighbour
MIN_CONTACTS = 20                               # below this it is a tip touch,
                                                # not a stacked pair
CELLS = (-2, -1, 0, 1, 2)                       # translations of the unit cell
                                                # built around the origin

if not os.path.exists(CIF):
    raise SystemExit(f"{CIF} not found in {os.getcwd()} - download the "
                     f"structure from CCDC entry 2161927 and put it here")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"reading {CIF}, writing to {OUTPUT_DIR}/")

In [ ]:
# ---- Helpers ------------------------------------------------------------
def num(tok):
    """A CIF number, dropping the parenthesised standard uncertainty."""
    return float(re.sub(r"\(\d+\)", "", tok))


def parse_cif(path):
    """Cell parameters, symmetry operations, and the asymmetric unit.

    Written out rather than taken from a library so the only dependencies are
    numpy and scipy: CIF loops are simple enough to walk directly.
    """
    lines = open(path, encoding="utf-8", errors="replace").read().splitlines()
    cell, symops, atoms, i = {}, [], [], 0
    while i < len(lines):
        ln = lines[i].strip()
        for key in ("_cell_length_a", "_cell_length_b", "_cell_length_c",
                    "_cell_angle_alpha", "_cell_angle_beta",
                    "_cell_angle_gamma"):
            if ln.startswith(key + " "):
                cell[key] = num(ln.split()[1])
        if ln == "loop_":
            j, names = i + 1, []
            while j < len(lines) and lines[j].strip().startswith("_"):
                names.append(lines[j].strip())
                j += 1
            body = []
            while j < len(lines):
                s = lines[j].strip()
                if not s or s.startswith(("_", "#")) or s == "loop_":
                    break
                body.append(s)
                j += 1
            if "_space_group_symop_operation_xyz" in names:
                symops += [b.strip("'\" ") for b in body]
            if "_atom_site_label" in names and "_atom_site_fract_x" in names:
                k = {n: c for c, n in enumerate(names)}
                for b in body:
                    t = b.split()
                    if len(t) < len(names):
                        continue
                    atoms.append({
                        "label": t[k["_atom_site_label"]],
                        "sym": t[k["_atom_site_type_symbol"]],
                        "f": np.array([num(t[k["_atom_site_fract_x"]]),
                                       num(t[k["_atom_site_fract_y"]]),
                                       num(t[k["_atom_site_fract_z"]])]),
                        "grp": t[k["_atom_site_disorder_group"]]})
            i = j
            continue
        i += 1
    return cell, symops, atoms


def cart_matrix(cell):
    """Fractional to Cartesian, for a general triclinic cell."""
    a, b, c = (cell["_cell_length_a"], cell["_cell_length_b"],
               cell["_cell_length_c"])
    al, be, ga = (np.radians(cell["_cell_angle_alpha"]),
                  np.radians(cell["_cell_angle_beta"]),
                  np.radians(cell["_cell_angle_gamma"]))
    v = np.sqrt(1 - np.cos(al)**2 - np.cos(be)**2 - np.cos(ga)**2
                + 2 * np.cos(al) * np.cos(be) * np.cos(ga))
    return np.array([[a, b * np.cos(ga), c * np.cos(be)],
                     [0, b * np.sin(ga),
                      c * (np.cos(al) - np.cos(be) * np.cos(ga)) / np.sin(ga)],
                     [0, 0, c * v / np.sin(ga)]])


def apply_op(op, f):
    """Apply one symmetry operation, written in the CIF as e.g. '-x, y+1/2, -z'."""
    x, y, z = f
    return np.array([eval(t, {"x": x, "y": y, "z": z}) for t in op.split(",")])


def core_number(label):
    d = "".join(c for c in label[1:] if c.isdigit())
    return int(d) if d else 0


def is_core(label, sym):
    """Conjugated core: everything but the alkyl chains (C24 and above)."""
    if sym == "H":
        return False
    return sym != "C" or core_number(label) <= 23


def is_end_group(label):
    """The indanone / dicyanomethylene terminal unit. The fraction of contact
    atoms that belong to it is what separates the J motif from the H motif."""
    if label.startswith(("F", "O")):
        return True
    if label.startswith("N"):
        return label in ("N3", "N4")
    return label.startswith("C") and 12 <= core_number(label) <= 23


def write_xyz(path, syms, coords, comment):
    with open(path, "w") as f:
        f.write(f"{len(syms)}\n{comment}\n")
        for s, r in zip(syms, coords):
            f.write(f"{s:2s}  {r[0]:12.6f}  {r[1]:12.6f}  {r[2]:12.6f}\n")

## Step 1: the ordered part of the asymmetric unit

Solvent and the minor disorder components are dropped: they are not part of the
molecule and they confuse the connectivity clustering later.

In [ ]:
# ---- Step 1: read the ordered part of the asymmetric unit ---------------
cell, symops, atoms = parse_cif(CIF)
keep = [a for a in atoms if a["label"] not in SOLVENT and a["grp"] not in MINOR]
M = cart_matrix(cell)
Minv = np.linalg.inv(M)
print(f"{len(symops)} symmetry operations; asymmetric unit {len(keep)} atoms "
      f"after dropping solvent and minor disorder components")

## Step 2: keep the core, cap where the chains were

Every heavy atom of the conjugated core is kept, along with the hydrogens bonded
to it. Where a chain used to attach, a hydrogen is placed along the broken bond
at a normal C-H or N-H length.

In [ ]:
# ---- Step 2: keep the core, cap where the chains were -------------------
p_au = np.array([M @ a["f"] for a in keep])
s_au = [a["sym"] for a in keep]
l_au = [a["label"] for a in keep]
iscore = np.array([is_core(l, s) for l, s in zip(l_au, s_au)])
tree = cKDTree(p_au)

fpos, fsym, flab = [], [], []
for i in range(len(p_au)):
    if s_au[i] == "H":
        nb = [j for j in tree.query_ball_point(p_au[i], 1.3)
              if j != i and s_au[j] != "H"]
        if not nb or not iscore[nb[0]]:
            continue
    elif not iscore[i]:
        continue
    fpos.append(p_au[i])
    fsym.append(s_au[i])
    flab.append(l_au[i])

ncap = 0
for i in range(len(p_au)):
    if not iscore[i]:
        continue
    for j in tree.query_ball_point(p_au[i], 2.1):
        if j == i or s_au[j] == "H" or iscore[j]:
            continue
        v = p_au[j] - p_au[i]
        fpos.append(p_au[i] + CAP[s_au[i]] * v / np.linalg.norm(v))
        fsym.append("H")
        flab.append(f"Hcap{l_au[i]}")
        ncap += 1
print(f"capped {ncap} chain attachment points; "
      f"fragment {len(fpos)} atoms {dict(sorted(Counter(fsym).items()))}")

## Step 3: build the packing and cluster it into molecules

Applying every symmetry operation over a 5x5x5 block of cells gives the local
packing. Atoms are then joined into molecules by covalent connectivity, and only
complete molecules - those with the full atom count - are kept, so fragments cut
by the edge of the block cannot be mistaken for neighbours.

In [ ]:
# ---- Step 3: build the packing and cluster into molecules ---------------
frac = [Minv @ p for p in fpos]
pos, sym_, lab = [], [], []
for op in symops:
    for ta in CELLS:
        for tb in CELLS:
            for tc in CELLS:
                for f, s, l in zip(frac, fsym, flab):
                    pos.append(M @ (apply_op(op, f) + np.array([ta, tb, tc])))
                    sym_.append(s)
                    lab.append(l)
pos, sym_, lab = np.array(pos), np.array(sym_), np.array(lab)

tree = cKDTree(pos)
pairs = tree.query_pairs(2.2, output_type="ndarray")
radii = np.array([RCOV[s] for s in sym_])
d = np.linalg.norm(pos[pairs[:, 0]] - pos[pairs[:, 1]], axis=1)
pairs = pairs[d < 1.15 * (radii[pairs[:, 0]] + radii[pairs[:, 1]])]

parent = np.arange(len(pos))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


for i, j in pairs:
    ri, rj = find(i), find(j)
    if ri != rj:
        parent[ri] = rj
roots = np.array([find(i) for i in range(len(pos))])
sizes = Counter(roots)
full = max(sizes.values())
mols = [np.where(roots == r)[0] for r, n in sizes.items() if n == full]
print(f"{len(mols)} complete molecules of {full} atoms in the block")

## Step 4: the neighbour shell, and the two motifs

Take the most interior molecule, find every neighbour it touches within 4.5 A,
and describe each contact by its closest approach, its interplanar spacing, and
how many of the contacting atoms belong to the end groups. That last fraction is
what separates the two motifs: a contact made mostly through the terminal
indanone/dicyanomethylene units is the J motif, one made through the backbone is
the H motif.

Symmetry-equivalent neighbours are collapsed, and contacts with fewer than 20
atom pairs are reported but not written - those are tip touches, not stacks.

Only the `_dimer.xyz` files are saved, since that is all stage 2 reads.

In [ ]:
# ---- Step 4: the neighbour shell of the most interior molecule ---------
cents = np.array([pos[m].mean(axis=0) for m in mols])
ref = mols[int(np.argmin(np.linalg.norm(cents - cents.mean(axis=0), axis=1)))]
ha = ref[sym_[ref] != "H"]

rows = []
for m in mols:
    if np.array_equal(m, ref):
        continue
    hb = m[sym_[m] != "H"]
    dm = np.linalg.norm(pos[ha][:, None, :] - pos[hb][None, :, :], axis=-1)
    if dm.min() > CONTACT:
        continue
    ca = ha[dm.min(axis=1) < CONTACT]
    cb = hb[dm.min(axis=0) < CONTACT]
    if len(ca) >= 8 and len(cb) >= 8:
        c0 = pos[ca].mean(axis=0)
        _, _, vt = np.linalg.svd(pos[ca] - c0)
        inter = np.abs((pos[cb] - c0) @ vt[2]).mean()
    else:
        inter = float("nan")
    rows.append({"m": m, "min": dm.min(), "inter": inter,
                 "n": int((dm < CONTACT).sum()),
                 "end": np.mean([is_end_group(l)
                                 for l in lab[np.concatenate([ca, cb])]])})

rows.sort(key=lambda r: -r["n"])
seen, unique = {}, []
for r in rows:
    key = (round(r["min"], 2), round(r["n"], -1))
    if key in seen:
        seen[key][0] += 1
        continue
    seen[key] = [1, r]
    unique.append(r)

print(f"{len(rows)} neighbours within {CONTACT} A\n")
print(f"{'min':>6} {'interplanar':>12} {'contacts':>9} {'end%':>6} "
      f"{'x':>3}  motif")
written = []
for r in unique:
    mult = seen[(round(r["min"], 2), round(r["n"], -1))][0]
    motif = "J_end_group" if r["end"] > 0.6 else "H_backbone"
    stacked = r["n"] >= MIN_CONTACTS
    print(f"{r['min']:6.2f} {r['inter']:12.2f} {r['n']:9d} "
          f"{r['end'] * 100:5.0f}% {mult:3d}  "
          f"{motif if stacked else 'peripheral contact'}")
    if not stacked:
        continue
    q = np.vstack([pos[ref], pos[r["m"]]])
    s = np.concatenate([sym_[ref], sym_[r["m"]]])
    note = (f"AQx-2 {motif} dimer from CCDC 2161927: closest contact "
            f"{r['min']:.3f} A, interplanar {r['inter']:.3f} A, "
            f"{r['n']} contacts, multiplicity {mult}, n_A={len(ref)}")
    path = f"{OUTPUT_DIR}/{motif}_dimer.xyz"
    write_xyz(path, s, q, note)
    written.append(path)

print(f"\nwrote {len(written)} dimers ({len(ref)} atoms per monomer)")
print("published values for comparison: J-aggregation 3.341 A between end "
      "groups,\n                                 H-aggregation 3.424 A "
      "between backbones")

## Step 5: check the files stage 2 will read

Stage 2 splits each dimer down the middle and requires the two halves to be the
same molecule in the same atom order. Anything wrong with that shows up here
rather than an hour into an MD run.

In [ ]:
# ---- Step 5: verify the outputs -----------------------------------------
def load_xyz(path):
    L = open(path).read().splitlines()
    n = int(L[0].split()[0])
    sym = [ln.split()[0] for ln in L[2:2 + n]]
    p = np.array([[float(x) for x in ln.split()[1:4]] for ln in L[2:2 + n]])
    return sym, p, L[1]


for path in sorted(written):
    sym, p, head = load_xyz(path)
    nA = len(sym) // 2
    if len(sym) % 2 or sym[:nA] != sym[nA:]:
        raise SystemExit(f"{path}: the two halves are not the same molecule "
                         f"in the same order - stage 2 will refuse this file")
    heavy = np.array([i for i, s in enumerate(sym[:nA]) if s != "H"])
    a, b = p[:nA][heavy], p[nA:][heavy]
    ca, cb = a.mean(0), b.mean(0)
    _, _, vt = np.linalg.svd(a - ca)
    delta = cb - ca
    d = np.linalg.norm(p[:nA, None, :] - p[None, nA:, :], axis=-1)
    print(f"{path}: {len(sym)} atoms ({nA} per monomer)")
    print(f"    separation {abs(delta @ vt[2]):.2f} A, slip "
          f"{delta @ vt[0]:.2f} / {delta @ vt[1]:.2f} A, COM "
          f"{np.linalg.norm(delta):.2f} A")
    print(f"    closest contact {d.min():.2f} A, overlap "
          f"{(d.min(axis=1) < 5.0).mean() * 100:.0f}%")

print("\nready for stage 2: set MOTIF to either name and run "
      "generate_configs.ipynb")